In [ ]:
# Cell 1: Install dependencies
!pip install pydub beautifulsoup4 requests
!apt-get -qq install -y ffmpeg

In [ ]:
# Cell 2: Clone repo and set configuration
# IMPORTANT: Push your local commits BEFORE running this cell!
!git clone https://github.com/bibleman-stan/readers-bofm.git
%cd readers-bofm

# ── Configuration ──
ELEVENLABS_API_KEY = "YOUR_ELEVENLABS_API_KEY_HERE"
VOICE_ID = "iJpuSZDxZSBJHFSeavba"   # Sister M
VOICE_NAME = "Sister M"
MODEL_ID = "eleven_multilingual_v2"

LINE_PAUSE_MS = 180     # 180ms between sense-lines
VERSE_PAUSE_MS = 500    # 500ms between verses

print("Config set!")
print(f"  Voice: {VOICE_NAME} ({VOICE_ID})")
print(f"  Model: {MODEL_ID}")

In [ ]:
# Cell 3: Core functions — HTML parsing + ElevenLabs TTS + Line Caching
import re
import json
import time
import hashlib
import requests
import io
import os
from pathlib import Path
from bs4 import BeautifulSoup, NavigableString
from pydub import AudioSegment

# ── Voice Settings ──
VOICE_SETTINGS = {
    "stability": 0.50,
    "similarity_boost": 0.75,
    "style": 0.35,
    "use_speaker_boost": True,
}

# ── Cache directory for per-line audio ──
CACHE_DIR = Path('audio_cache')
CACHE_DIR.mkdir(exist_ok=True)

def cache_key(text, voice_id, model_id, settings):
    blob = json.dumps({"text": text, "voice_id": voice_id, "model_id": model_id, "settings": settings}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

def get_cached_audio(key):
    path = CACHE_DIR / f"{key}.mp3"
    if path.exists():
        return AudioSegment.from_mp3(path)
    return None

def save_to_cache(key, audio_seg):
    path = CACHE_DIR / f"{key}.mp3"
    audio_seg.export(str(path), format="mp3", bitrate="128k")


def get_text_from_element(el, use_modern=False):
    clone = BeautifulSoup(str(el), 'html.parser')
    if use_modern:
        for swap in clone.find_all(class_=['swap', 'swap-quiet']):
            mod = swap.get('data-mod')
            if mod:
                swap.string = mod
    for remove in clone.find_all(class_=['verse-num', 'parry-label-spacer', 'parry-label']):
        remove.decompose()
    text = clone.get_text()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def parse_chapter(html_path, book_id, chapter_num):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    chapter_id = f'ch-{book_id}-{chapter_num}'
    chapter_div = soup.find(id=chapter_id)
    if not chapter_div:
        print(f'  Warning: chapter {chapter_id} not found')
        return []
    items = []
    line_idx = 0
    for child in chapter_div.children:
        if isinstance(child, NavigableString):
            continue
        classes = child.get('class', [])
        if 'pericope-header' in classes:
            line_idx += 1
            continue
        if 'verse' in classes:
            verse_num_el = child.find(class_='verse-num')
            vn = verse_num_el.get_text().strip() if verse_num_el else ''
            sense_lines = [el for el in child.find_all('span', recursive=False) if 'line' in (el.get('class') or [])]
            for line_el in sense_lines:
                text = get_text_from_element(line_el, use_modern=False)
                if text:
                    items.append({'type': 'line', 'text': text, 'verse_num': vn, 'line_index': line_idx})
                    line_idx += 1
            if items and items[-1]['type'] != 'verse-gap':
                items.append({'type': 'verse-gap', 'text': '', 'verse_num': '', 'line_index': -1})
    return items


def tts_elevenlabs(text, voice_id=VOICE_ID, model_id=MODEL_ID, settings=None):
    if settings is None:
        settings = VOICE_SETTINGS
    key = cache_key(text, voice_id, model_id, settings)
    cached = get_cached_audio(key)
    if cached is not None:
        return cached, True
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    headers = {"xi-api-key": ELEVENLABS_API_KEY, "Content-Type": "application/json", "Accept": "audio/mpeg"}
    payload = {"text": text, "model_id": model_id, "voice_settings": settings}
    resp = requests.post(url, json=payload, headers=headers)
    if resp.status_code != 200:
        print(f"  API error {resp.status_code}: {resp.text[:200]}")
        return AudioSegment.silent(duration=500), False
    audio_seg = AudioSegment.from_mp3(io.BytesIO(resp.content))
    save_to_cache(key, audio_seg)
    return audio_seg, False


def generate_silence_seg(duration_ms):
    return AudioSegment.silent(duration=duration_ms)


def stitch_chapter(items, voice_id=VOICE_ID, model_id=MODEL_ID,
                   settings=None, line_pause_ms=LINE_PAUSE_MS,
                   verse_pause_ms=VERSE_PAUSE_MS, stitch_only=False):
    if settings is None:
        settings = VOICE_SETTINGS
    combined = AudioSegment.empty()
    timing = []
    current_time_ms = 0
    stats = {"cached": 0, "generated": 0, "skipped": 0, "chars_used": 0}
    speakable = [i for i in items if i['type'] == 'line']
    total = len(speakable)
    done = 0
    for item in items:
        if item['type'] == 'verse-gap':
            combined += generate_silence_seg(verse_pause_ms)
            current_time_ms += verse_pause_ms
            continue
        if item['type'] != 'line':
            continue
        done += 1
        key = cache_key(item['text'], voice_id, model_id, settings)
        if stitch_only:
            cached = get_cached_audio(key)
            if cached is None:
                print(f"  SKIP [{done}/{total}] v{item['verse_num']}: not in cache")
                stats["skipped"] += 1
                continue
            audio_seg = cached
            stats["cached"] += 1
            tag = "cache"
        else:
            audio_seg, from_cache = tts_elevenlabs(item['text'], voice_id, model_id, settings)
            if from_cache:
                stats["cached"] += 1
                tag = "cache"
            else:
                stats["generated"] += 1
                stats["chars_used"] += len(item['text'])
                tag = "NEW"
                time.sleep(0.05)
        start_ms = current_time_ms
        combined += audio_seg
        current_time_ms += len(audio_seg)
        timing.append({
            'start': round(start_ms / 1000, 3),
            'end': round(current_time_ms / 1000, 3),
            'type': 'line',
            'text': item['text'],
            'verse': item['verse_num'],
            'lineIndex': item['line_index'],
        })
        print(f"  [{done}/{total}] [{tag}] v{item['verse_num']}: {item['text'][:50]}...")
        combined += generate_silence_seg(line_pause_ms)
        current_time_ms += line_pause_ms
    return combined, timing, stats


print("Core functions loaded!")
print(f"  Cache dir: {CACHE_DIR}")
print(f"  Voice settings: stability={VOICE_SETTINGS['stability']}, similarity={VOICE_SETTINGS['similarity_boost']}, style={VOICE_SETTINGS['style']}")

In [ ]:
# Cell 4: VOICE TEST — Sister M on 5 lines from 2 Nephi 1
# Listen to this BEFORE running the full generation!
# Only costs ~250 chars. If you don't like the voice, stop here.
from IPython.display import Audio, display

SISTER_M_ID = "iJpuSZDxZSBJHFSeavba"

items = parse_chapter('books/2nephi.html', '2nephi', 1)
speakable = [i for i in items if i['type'] == 'line']
test_items = speakable[:5]

print(f"Voice test (Sister M): {len(test_items)} lines from 2 Nephi 1")
for i, item in enumerate(test_items):
    print(f"  {i+1}. [{item['verse_num']}] {item['text'][:70]}...")

total_chars = sum(len(item['text']) for item in test_items)
print(f"\nChars for test: ~{total_chars}")

print(f"\n{'='*40}")
print(f"Generating Sister M test: {SISTER_M_ID}")
print(f"{'='*40}")
combined_test = AudioSegment.empty()
for i, item in enumerate(test_items):
    print(f"  [{i+1}/5] {item['text'][:50]}...")
    seg, cached = tts_elevenlabs(item['text'], voice_id=SISTER_M_ID)
    tag = "cache" if cached else "NEW"
    print(f"         [{tag}]")
    combined_test += seg
    combined_test += generate_silence_seg(LINE_PAUSE_MS)
    if not cached:
        time.sleep(0.1)

combined_test.export("test_sister_m.mp3", format="mp3", bitrate="128k")
print(f"\nSister M test: {len(combined_test)/1000:.1f}s")
display(Audio("test_sister_m.mp3"))

In [ ]:
# Cell 5: FULL GENERATION — 2 Nephi (all 33 chapters) with Sister M
# ~197k characters, 4167 lines across 33 chapters
# Per-line caching: re-running is cheap (cached lines skip API)

BOOK_ID = '2nephi'
BOOK_NAME = '2 Nephi'
TOTAL_CHAPTERS = 33
SISTER_M_ID = "iJpuSZDxZSBJHFSeavba"

Path('audio').mkdir(exist_ok=True)
results = []

for ch in range(1, TOTAL_CHAPTERS + 1):
    print(f"\n{'='*60}")
    print(f"CHAPTER {ch} of {TOTAL_CHAPTERS}")
    print(f"{'='*60}")

    items = parse_chapter(f'books/{BOOK_ID}.html', BOOK_ID, ch)
    speakable = [i for i in items if i['type'] == 'line']
    total_chars = sum(len(i['text']) for i in speakable)
    print(f"  {len(speakable)} lines, ~{total_chars} chars")

    combined, timing, stats = stitch_chapter(items, voice_id=SISTER_M_ID)

    # Export MP3
    mp3_path = f'audio/{BOOK_ID}-{ch}-sister-m.mp3'
    combined.export(mp3_path, format='mp3', bitrate='128k')
    duration_s = len(combined) / 1000
    file_size = os.path.getsize(mp3_path)

    # Export timing manifest
    json_path = f'audio/{BOOK_ID}-{ch}-sister-m.json'
    manifest = {
        'book': BOOK_ID,
        'bookName': BOOK_NAME,
        'chapter': ch,
        'voice': {
            'provider': 'elevenlabs',
            'voice_id': SISTER_M_ID,
            'model_id': MODEL_ID,
            'settings': VOICE_SETTINGS,
            'name': 'Sister M',
        },
        'pauses': {
            'line_ms': LINE_PAUSE_MS,
            'verse_ms': VERSE_PAUSE_MS,
        },
        'duration': round(duration_s, 3),
        'lines': timing,
    }
    with open(json_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    results.append({
        'ch': ch,
        'lines': len(speakable),
        'duration': duration_s,
        'size_kb': file_size / 1024,
        'cached': stats['cached'],
        'generated': stats['generated'],
        'chars_used': stats['chars_used'],
    })

    print(f"  MP3: {duration_s:.1f}s, {file_size/1024:.0f}KB")
    print(f"  Cache hits: {stats['cached']}, New: {stats['generated']}, Credits: ~{stats['chars_used']}")

# ── Summary ──
print(f"\n\n{'='*60}")
print(f"2 NEPHI COMPLETE — {TOTAL_CHAPTERS} chapters with Sister M")
print(f"{'='*60}")
total_duration = sum(r['duration'] for r in results)
total_size = sum(r['size_kb'] for r in results)
total_lines = sum(r['lines'] for r in results)
total_new = sum(r['generated'] for r in results)
total_cached = sum(r['cached'] for r in results)
total_chars = sum(r['chars_used'] for r in results)
print(f"  Total lines: {total_lines}")
print(f"  Total duration: {total_duration/60:.1f} min ({total_duration:.0f}s)")
print(f"  Total size: {total_size/1024:.1f} MB")
print(f"  Cache hits: {total_cached}, Newly generated: {total_new}")
print(f"  Credits used: ~{total_chars} chars")
for r in results:
    print(f"  Ch{r['ch']:2d}: {r['lines']:3d} lines, {r['duration']:6.1f}s, {r['size_kb']:6.0f}KB")

In [ ]:
# Cell 6: Download 2 Nephi Sister M audio files (zip)
import zipfile
from google.colab import files

zip_path = 'audio/2nephi_sister_m_audio.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ch in range(1, 34):
        mp3 = f'audio/2nephi-{ch}-sister-m.mp3'
        jsn = f'audio/2nephi-{ch}-sister-m.json'
        if os.path.exists(mp3):
            zf.write(mp3, os.path.basename(mp3))
        if os.path.exists(jsn):
            zf.write(jsn, os.path.basename(jsn))

zip_size = os.path.getsize(zip_path) / (1024*1024)
print(f"Zip created: {zip_path} ({zip_size:.1f} MB)")
print(f"Downloading...")
files.download(zip_path)